In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
import tensorflow as tf
import tensorflow_probability as tfp
import gpflow as gpf
from tqdm import tqdm

D:\Programmi\Anaconda3\envs\XAI_GPFlowTest\Lib\site-packages\gpflow\versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
df_train_noCS = pd.read_csv("data/train_noCS.csv")
if 'index' in df_train_noCS.columns:
    df_train_noCS = df_train_noCS.drop(['index'], axis=1)

df_test_noCS = pd.read_csv("data/test_noCS.csv")
if 'index' in df_test_noCS.columns:
    df_test_noCS = df_test_noCS.drop(['index'], axis=1)

def extract_yX(df):
    y = df['Z']
    X = df.drop(['Z'], axis=1)
    return y , X 

#Preparazione dei dati:
y_tr, X_tr = extract_yX(df_train_noCS)
y_te, X_te = extract_yX(df_test_noCS)

X_tr = X_tr.to_numpy().astype(np.float64)
y_tr = y_tr.to_numpy().astype(np.float64).reshape(-1, 1)
X_te = X_te.to_numpy().astype(np.float64)
y_te = y_te.to_numpy().astype(np.float64).reshape(-1, 1)


In [3]:
likelihood = gpf.likelihoods.HeteroskedasticTFPConditional(
    distribution_class=tfp.distributions.Normal,  # Gaussian Likelihood
    scale_transform=tfp.bijectors.Exp(),  # Exponential Transform
)

print(f"Likelihood's expected latent_dim: {likelihood.latent_dim}")


kernel = gpf.kernels.SeparateIndependent(
    [
        gpf.kernels.SquaredExponential(),  # This is k1, the kernel of f1
        gpf.kernels.SquaredExponential(),  # this is k2, the kernel of f2
    ]
)
# The number of kernels contained in gpf.kernels.SeparateIndependent must be the same as likelihood.latent_dim


M = 20  # Number of inducing variables for each f_i

# scegli M punti casuali dal training set (considerando che dataset ha più features)
indices = np.random.choice(len(X_tr), M, replace=False)
Z = X_tr[indices]

inducing_variable = gpf.inducing_variables.SeparateIndependentInducingVariables(
    [
        gpf.inducing_variables.InducingPoints(Z),  # This is U1 = f1(Z1)
        gpf.inducing_variables.InducingPoints(Z),  # This is U2 = f2(Z2)
    ]
)


model = gpf.models.SVGP(
    kernel=kernel,
    likelihood=likelihood,
    inducing_variable=inducing_variable,
    num_latent_gps=likelihood.latent_dim,
)

model


data = (X_tr, y_tr)
loss_fn = model.training_loss_closure(data)

gpf.utilities.set_trainable(model.q_mu, False)
gpf.utilities.set_trainable(model.q_sqrt, False)

variational_vars = [(model.q_mu, model.q_sqrt)]
natgrad_opt = gpf.optimizers.NaturalGradient(gamma=0.1)

adam_vars = model.trainable_variables
adam_opt = tf.optimizers.Adam(0.01)


@tf.function
def optimisation_step():
    natgrad_opt.minimize(loss_fn, variational_vars)

    with tf.GradientTape() as tape:
        loss = loss_fn()

    grads = tape.gradient(loss, adam_vars)
    adam_opt.apply_gradients(zip(grads, adam_vars))
    
    
epochs = 100
log_freq = 20

for epoch in tqdm(range(1, epochs + 1)):
    optimisation_step()

    # For every 'log_freq' epochs, print the epoch and plot the predictions against the data
    if epoch % log_freq == 0 and epoch > 0:
        print(f"Epoch {epoch} - Loss: {loss_fn().numpy() : .4f}")
        Ymean, Yvar = model.predict_y(X_tr)
        Ymean = Ymean.numpy().squeeze()
        Ystd = tf.sqrt(Yvar).numpy().squeeze()

model

Likelihood's expected latent_dim: 2


  0%|          | 0/100 [00:00<?, ?it/s]

Instructions for updating:
Use fn_output_signature instead


 19%|█▉        | 19/100 [00:37<01:18,  1.03it/s]

Epoch 20 - Loss:  10983.2651


...

Epoch 80 - Loss:  312.7396


 99%|█████████▉| 99/100 [02:15<00:01,  1.04s/it]

Epoch 100 - Loss: -876.9304


100%|██████████| 100/100 [02:17<00:00,  1.38s/it]


name,class,transform,prior,trainable,shape,dtype,value
SVGP.kernel.kernels[0].variance,Parameter,Softplus,,True,(),float64,0.544544060100935
SVGP.kernel.kernels[0].lengthscales,Parameter,Softplus,,True,(),float64,1.66389
SVGP.kernel.kernels[1].variance,Parameter,Softplus,,True,(),float64,0.7021273626119333
SVGP.kernel.kernels[1].lengthscales,Parameter,Softplus,,True,(),float64,1.52484
SVGP.inducing_variable.inducing_variable_list[0].Z,Parameter,Identity,,True,"(20, 6)",float64,"[[2.57054400e+01, 5.96441144e-01, 8.13692404e-01..."
SVGP.inducing_variable.inducing_variable_list[1].Z,Parameter,Identity,,True,"(20, 6)",float64,"[[25.43553, 0.66206697, 0.56609116..."
SVGP.q_mu,Parameter,Identity,,False,"(20, 2)",float64,"[[0.68537928, -1.82474..."
SVGP.q_sqrt,Parameter,FillTriangular,,False,"(2, 20, 20)",float64,"[[[5.06480658e-03, 0.00000000e+00, 0.00000000e+00..."


In [4]:
Ymean_te, Yvar_te = model.predict_y(X_te)

rmse = np.sqrt(np.mean((y_te - Ymean_te.numpy())**2))
mae = np.mean(np.abs(y_te - Ymean_te.numpy()))

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

Test RMSE: 0.2832
Test MAE: 0.1833
